In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import math
import copy

device = "cuda" if torch.cuda.is_available() else "cpu"

In [28]:
transform = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor()
])

train_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset,batch_size=128,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=128,shuffle=False)

In [29]:
class PatchEmbed(nn.Module):

    def __init__(self,img_size=32,patch_size=4,in_chans=1,embed_dim=128):

        super().__init__()

        self.proj = nn.Conv2d(
            in_chans,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self,x):

        x = self.proj(x)

        B,C,H,W = x.shape

        x = x.flatten(2).transpose(1,2)

        return x

In [30]:
class Block(nn.Module):

    def __init__(self,dim,heads):

        super().__init__()

        self.norm1 = nn.LayerNorm(dim)

        self.attn = nn.MultiheadAttention(dim,heads,batch_first=True)

        self.norm2 = nn.LayerNorm(dim)

        self.mlp = nn.Sequential(

            nn.Linear(dim,dim*4),
            nn.GELU(),
            nn.Linear(dim*4,dim)

        )

    def forward(self,x):

        h,_ = self.attn(self.norm1(x),self.norm1(x),self.norm1(x))

        x = x + h

        x = x + self.mlp(self.norm2(x))

        return x


In [31]:
class Encoder(nn.Module):

    def __init__(self,embed_dim=128):

        super().__init__()

        self.patch = PatchEmbed()

        self.pos = nn.Parameter(torch.randn(1,64,embed_dim))

        self.blocks = nn.ModuleList(

            [Block(embed_dim,4) for _ in range(6)]

        )

        self.norm = nn.LayerNorm(embed_dim)

    def forward(self,x):

        x = self.patch(x)

        x = x + self.pos

        for blk in self.blocks:

            x = blk(x)

        x = self.norm(x)

        return x

In [32]:
class Predictor(nn.Module):

    def __init__(self, embed_dim=128, num_patches=64):

        super().__init__()

        self.mask_token = nn.Parameter(torch.randn(1,1,embed_dim))

        # ✅ positional embeddings (CRITICAL)
        self.pos_embed = nn.Parameter(torch.randn(1,num_patches,embed_dim))

        self.blocks = nn.ModuleList(
            [Block(embed_dim,4) for _ in range(4)]
        )

        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x, context_idx, target_idx):

        B = x.shape[0]

        # ✅ add positional info to context tokens
        pos_c = self.pos_embed[:, context_idx]
        x = x + pos_c

        # ✅ create mask tokens with positional info
        mask_tokens = self.mask_token.repeat(B, len(target_idx), 1)
        pos_t = self.pos_embed[:, target_idx]

        mask_tokens = mask_tokens + pos_t

        # concat
        x = torch.cat([x, mask_tokens], dim=1)

        # transformer
        for blk in self.blocks:
            x = blk(x)

        x = self.norm(x)

        # return only predictions
        return x[:, -len(target_idx):]

In [33]:
def sample_masks(N,ratio=0.25):

    perm = torch.randperm(N)

    split = int(N*(1-ratio))

    context = perm[:split]

    target = perm[split:]

    return context,target

In [34]:
encoder = Encoder().to(device)

predictor = Predictor().to(device)

target_encoder = copy.deepcopy(encoder)

for p in target_encoder.parameters():
    p.requires_grad=False

In [35]:
optimizer = torch.optim.AdamW(

    list(encoder.parameters())+
    list(predictor.parameters()),
    lr=1e-4,
    weight_decay=1e-4

)

In [36]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=100
)

In [37]:
def update_target():

    m = 0.996

    for q,k in zip(encoder.parameters(),target_encoder.parameters()):

        k.data = m*k.data + (1-m)*q.data

In [38]:
epochs = 50

for epoch in range(epochs):

    loop = tqdm(train_loader)

    for img,_ in loop:

        img = img.to(device)

        with torch.no_grad():
            h = target_encoder(img)

        N = h.shape[1]

        context_idx, target_idx = sample_masks(N)

        context_idx = context_idx.to(device)
        target_idx = target_idx.to(device)

        h_target = h[:, target_idx]

        z = encoder(img)

        z_context = z[:, context_idx]

        # ✅ FIXED LINE
        pred = predictor(z_context, context_idx, target_idx)

        loss = F.smooth_l1_loss(pred, h_target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        update_target()

        loop.set_description(f"loss {loss.item():.4f}")

    scheduler.step()

loss 0.0178: 100%|██████████| 469/469 [00:37<00:00, 12.66it/s]


In [39]:
for p in encoder.parameters():
    p.requires_grad=False

classifier = nn.Linear(128,10).to(device)

opt = torch.optim.Adam(classifier.parameters(),lr=1e-3)

criterion = nn.CrossEntropyLoss()

for epoch in range(10):

    for img,label in train_loader:

        img = img.to(device)

        label = label.to(device)

        with torch.no_grad():

            z = encoder(img)

            z = z.mean(dim=1)

        pred = classifier(z)

        loss = criterion(pred,label)

        opt.zero_grad()

        loss.backward()

        opt.step()

In [40]:
encoder.eval()
classifier.eval()

correct = 0
total = 0

with torch.no_grad():

    for img,label in test_loader:

        img = img.to(device)

        label = label.to(device)

        z = encoder(img)

        z = z.mean(dim=1)

        pred = classifier(z)

        predicted = pred.argmax(dim=1)

        correct += (predicted==label).sum().item()

        total += label.size(0)

print("Linear Probe Accuracy:",100*correct/total)

Linear Probe Accuracy: 89.93
